# 🔍 Policy Validation

This notebook validates a trained policy against a dataset by comparing predicted actions to ground truth actions.

**What it does:**
1. Loads a trained policy checkpoint
2. Loads a validation dataset
3. Runs the policy on each sample to get predicted actions
4. Plots ground truth vs predicted actions for each action dimension

This helps you visually assess how well your policy has learned the task.

---
## 1. Configuration

Specify the paths to your checkpoint and dataset.

> **Action Required:** Update `CHECKPOINT_DIR` and `DATASET_DIR` below.

In [ ]:
import pathlib

# --- Paths ---
# TODO: Path to your trained policy checkpoint
CHECKPOINT_DIR = pathlib.Path("/data/models/your_checkpoint_here")

# TODO: Path to the dataset to validate against
# This can be the same dataset you trained on, or a held-out validation set
DATASET_DIR = pathlib.Path("/data/lerobot/your_dataset_here")

# --- Validation Settings ---
# Maximum number of episodes to validate (set to None for all)
MAX_EPISODES = 10

# Output path for the plot (None = display inline only)
OUTPUT_PATH = None  # e.g., pathlib.Path("validation_plot.png")

print(f"Checkpoint: {CHECKPOINT_DIR}")
print(f"Dataset: {DATASET_DIR}")
print(f"Max episodes: {MAX_EPISODES if MAX_EPISODES else 'All'}")

---
## 2. Load Policy and Dataset

In [ ]:
import torch
from lerobot.datasets.lerobot_dataset import LeRobotDataset
from example_policies.robot_deploy.deploy_core.policy_loader import load_policy

# Select device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Load policy
print(f"\nLoading policy from {CHECKPOINT_DIR}...")
policy, cfg = load_policy(CHECKPOINT_DIR)
policy.to(device)
policy.eval()
print("✅ Policy loaded successfully!")

# Load dataset
print(f"\nLoading dataset from {DATASET_DIR}...")
dataset = LeRobotDataset(
    repo_id=str(DATASET_DIR),
    root=DATASET_DIR,
    video_backend="pyav",  # pyav works on JupyterHub, unlike torchcodec
)
print(f"✅ Dataset loaded: {dataset.meta.total_episodes} episodes, {dataset.meta.total_frames} frames")

---
## 3. Run Validation

Run the policy on the dataset and collect predictions vs ground truth.

In [ ]:
from tqdm.auto import tqdm

def to_device_batch(batch: dict, device: torch.device, non_blocking: bool = True):
    """Move all tensors in a batch to the specified device."""
    out = {}
    for k, v in batch.items():
        if torch.is_tensor(v):
            out[k] = v.to(device, non_blocking=non_blocking)
        else:
            out[k] = v
    return out

# Create dataloader
dataloader = torch.utils.data.DataLoader(
    dataset,
    num_workers=4,
    batch_size=1,
    shuffle=False,
    pin_memory=device != "cpu",
    drop_last=False,
)

# Dictionary to store data for each episode
episodes_data = {}
action_dim = None

print(f"Running validation on up to {MAX_EPISODES if MAX_EPISODES else 'all'} episodes...")
prev_ep = 0
# Collect predictions
with torch.no_grad():
    for batch in tqdm(dataloader, desc="Validating"):
        # Get episode index
        b_ep = batch.get("episode_index")
        if b_ep is None:
            raise KeyError("Expected key 'episode_index' in batch.")
        b_ep = int(b_ep.view(-1)[0].item())
        
        # Skip if we've reached max episodes
        if MAX_EPISODES is not None and b_ep >= MAX_EPISODES:
            break

        if prev_ep < b_ep:
            policy.reset()
            prev_ep += 1
        
        # Initialize episode data structure if needed
        if b_ep not in episodes_data:
            episodes_data[b_ep] = {
                "targets": [],
                "preds": [],
                "times": [],
            }
        
        # Move batch to device
        batch = to_device_batch(batch, device, non_blocking=True)
        
        # Get ground truth action
        tgt = batch["action"].detach().float().view(-1)
        if action_dim is None:
            action_dim = tgt.numel()
        
        # Get predicted action
        action = policy.select_action(batch)
        pred = action.detach().float().view(-1)
        
        # Store data
        episodes_data[b_ep]["targets"].append(tgt.cpu())
        episodes_data[b_ep]["preds"].append(pred.cpu())
        
        # Store timestamp
        t = float(batch["timestamp"].view(-1)[0].detach().cpu().item())
        episodes_data[b_ep]["times"].append(t)

# Stack data for each episode
for ep_idx in episodes_data:
    episodes_data[ep_idx]["targets"] = torch.stack(episodes_data[ep_idx]["targets"], dim=0)
    episodes_data[ep_idx]["preds"] = torch.stack(episodes_data[ep_idx]["preds"], dim=0)
    episodes_data[ep_idx]["times"] = torch.tensor(episodes_data[ep_idx]["times"])

num_episodes = len(episodes_data)
print(f"\n✅ Validated {num_episodes} episodes")

---
## 4. Compute Metrics

Calculate error metrics to quantify policy performance.

In [ ]:
import numpy as np

print("Validation Metrics per Episode:")
print("=" * 60)

all_mse = []
all_mae = []

for ep_idx in sorted(episodes_data.keys()):
    ep_data = episodes_data[ep_idx]
    targets = ep_data["targets"].numpy()
    preds = ep_data["preds"].numpy()
    
    # Calculate MSE and MAE
    mse = np.mean((targets - preds) ** 2)
    mae = np.mean(np.abs(targets - preds))
    
    all_mse.append(mse)
    all_mae.append(mae)
    
    print(f"Episode {ep_idx:3d}: MSE = {mse:.6f}, MAE = {mae:.6f}, Frames = {len(targets)}")

print("=" * 60)
print(f"Overall:     MSE = {np.mean(all_mse):.6f}, MAE = {np.mean(all_mae):.6f}")

---
## 5. Plot Results

Visualize ground truth vs predicted actions for each action dimension.

In [ ]:
import matplotlib.pyplot as plt

if num_episodes == 0:
    print("No episodes found in dataset.")
else:
    # Get action dimension from first episode
    first_ep = list(episodes_data.keys())[0]
    D = episodes_data[first_ep]["targets"].shape[1]
    
    # Create plots
    fig, axes = plt.subplots(D, 1, figsize=(12, 2.5 * D), sharex=True)
    if D == 1:
        axes = [axes]
    
    # Plot each episode
    for ep_idx in sorted(episodes_data.keys()):
        ep_data = episodes_data[ep_idx]
        times = ep_data["times"].numpy()
        targets = ep_data["targets"].numpy()
        preds = ep_data["preds"].numpy()
        
        # Use matplotlib's default color cycle
        color = f"C{ep_idx % 10}"
        
        for d in range(D):
            ax = axes[d]
            ax.plot(times, targets[:, d], color=color, alpha=0.7, 
                    label=f"Ep {ep_idx} Target", linestyle='-')
            ax.plot(times, preds[:, d], color=color, alpha=0.7, 
                    label=f"Ep {ep_idx} Pred", linestyle='--')
            ax.set_ylabel(f"dim {d}")
            ax.grid(True, linestyle="--", alpha=0.3)
            if d == 0:
                ax.set_title(f"Policy Validation: Ground Truth vs Predictions ({num_episodes} episodes)")
    
    axes[-1].set_xlabel("Time (s)")
    
    # Add legend
    if num_episodes <= 3:
        axes[0].legend(loc="upper right", fontsize='small')
    else:
        handles, labels = axes[0].get_legend_handles_labels()
        fig.legend(handles[:6], labels[:6], loc="upper right", fontsize='small')
    
    plt.tight_layout(rect=[0, 0, 0.98, 0.98])
    
    # Save if output path specified
    if OUTPUT_PATH:
        fig.savefig(OUTPUT_PATH, dpi=150, bbox_inches='tight')
        print(f"\n💾 Saved plot to {OUTPUT_PATH}")
    
    plt.show()

---
## 6. Per-Dimension Analysis (Optional)

Analyze error distribution for each action dimension.

In [ ]:
# Concatenate all episode data
all_targets = torch.cat([episodes_data[ep]["targets"] for ep in sorted(episodes_data.keys())], dim=0).numpy()
all_preds = torch.cat([episodes_data[ep]["preds"] for ep in sorted(episodes_data.keys())], dim=0).numpy()

errors = all_targets - all_preds

print("Per-Dimension Error Statistics:")
print("=" * 70)
print(f"{'Dim':<6} {'Mean Error':<15} {'Std Error':<15} {'MAE':<15} {'Max |Error|':<15}")
print("-" * 70)

for d in range(errors.shape[1]):
    dim_errors = errors[:, d]
    print(f"{d:<6} {np.mean(dim_errors):<15.6f} {np.std(dim_errors):<15.6f} "
          f"{np.mean(np.abs(dim_errors)):<15.6f} {np.max(np.abs(dim_errors)):<15.6f}")

print("=" * 70)

In [ ]:
# Error distribution histograms
D = errors.shape[1]
fig, axes = plt.subplots(1, D, figsize=(3 * D, 3))
if D == 1:
    axes = [axes]

for d in range(D):
    ax = axes[d]
    ax.hist(errors[:, d], bins=50, edgecolor='black', alpha=0.7)
    ax.axvline(x=0, color='red', linestyle='--', linewidth=1)
    ax.set_xlabel('Error')
    ax.set_ylabel('Count')
    ax.set_title(f'Dim {d}')
    ax.grid(True, alpha=0.3)

plt.suptitle('Error Distribution per Action Dimension', fontsize=14)
plt.tight_layout()
plt.show()

---
## ✅ Done!

**Interpreting Results:**

- **Low MSE/MAE**: The policy accurately predicts the demonstrated actions
- **High MSE/MAE**: The policy may need more training or the task is challenging
- **Bias in error**: If mean error ≠ 0, the policy has a systematic bias
- **High variance in specific dimensions**: Some action components may be harder to learn

**Note:** Good validation metrics don't guarantee good real-world performance. Always test on the actual robot!